# CLO tranche modeling (two-factor intuition)

**Start here:** This deep dive expands on `07_advanced_quant/correlation_and_credit_models.ipynb`; read the overview first for the common copula, latent-factor, and recovery vocabulary.

Use `LatentTwoFactor.clo_standard()` to map prepay/credit factors into the canonical finite-pool credit-loss simulation, then present subordination and expected tranche loss from the Rust-owned pool loss distribution.


## Concept

**Base correlation** is a market quoting convention: implied correlation is backed out per tranche so a model reproduces observed premiums. Economically, senior tranches are protected by subordination: they take losses only after junior attachment points are exhausted.

The CLO-standard two-factor model maps independent normals $(Z_1,Z_2)$ to the credit state $Z_c=L_{10}Z_1+L_{11}Z_2$. A name with asset correlation $\rho$ therefore uses portfolio-engine loadings $\sqrt{\rho}[L_{10},L_{11}]$. The Rust engine draws factors and idiosyncratic shocks, tests defaults, and returns the finite-pool loss distribution.

This notebook does not reprice a CLO. Both the pool loss simulation *and* the attachment/detachment loss allocation are Rust-owned; the notebook only defines the capital structure and formats the results.

## API walkthrough

- **`LatentTwoFactor.clo_standard()`** supplies the Cholesky coefficients for $Z_c$.
- **`CreditExposure.factor_loadings`** receives $\sqrt{\rho}[L_{10},L_{11}]$ for each homogeneous name.
- **`PortfolioLossConfig(..., CopulaSpec.gaussian())`** selects deterministic path count, seed, confidence, and copula.
- **`simulate_portfolio_loss`** returns path losses, expected loss, loss-positive VaR, and expected shortfall.
- **`PortfolioLossResult.tranche_loss_statistics(attachment, detachment, pool_notional)`** allocates every path loss to one tranche and aggregates it, returning a `TrancheLossStatistics`.
- A constant **`RecoverySpec`** supplies the pool LGD used on each exposure.

In [1]:
from finstack_quant.valuations.correlation import (
    CopulaSpec,
    CreditExposure,
    LatentTwoFactor,
    PortfolioLossConfig,
    RecoverySpec,
    simulate_portfolio_loss,
)

tf = LatentTwoFactor.clo_standard()
rec = RecoverySpec.constant(0.40).build()
print(f"CLO standard: {tf}")
print(f"Cholesky credit map: Z_c = {tf.cholesky_l10:.4f}*Z1 + {tf.cholesky_l11:.4f}*Z2")
print(f"Recovery model: {rec}")

CLO standard: LatentTwoFactor(prepay=0.1500, credit=0.3000, corr=-0.2000)
Cholesky credit map: Z_c = -0.2000*Z1 + 0.9798*Z2
Recovery model: RecoveryModel('Constant Recovery', expected=0.4000)


### Capital structure and the units convention

For each simulated pool loss fraction $\ell$, the tranche loss fraction between attachment $A$ and detachment $D$ is

$$L^{\mathrm{tr}} = \frac{\min(\max(\ell - A, 0),\, D-A)}{D-A}$$

This allocation is owned by Rust: `PortfolioLossResult.tranche_loss_statistics(attachment, detachment, pool_notional)` applies it to every simulated path and aggregates the resulting distribution at the result's own confidence level.

**Units:** `attachment` and `detachment` are **fractions of pool notional in $[0, 1]$**, not percentages. A 0–3% equity tranche is `(0.0, 0.03)` — passing `(0.0, 3.0)` would silently define a tranche three times the size of the pool. The cell below defines the stack once and asserts that convention.

In [2]:
# Attachment/detachment points are FRACTIONS of pool notional in [0, 1] — never percentages.
tranches = [
    ("equity", 0.0, 0.03),
    ("mezz", 0.03, 0.07),
    ("senior", 0.07, 0.15),
    ("super_senior", 0.15, 1.0),
]

assert all(0.0 <= attach < detach <= 1.0 for _, attach, detach in tranches), "attach/detach must be fractions in [0, 1]"

print("Capital structure (fractions of pool notional):")
for label, attach, detach in tranches:
    print(f"  {label:<14} attach={attach:<6.4f} detach={detach:<6.4f}  ({attach:.0%}-{detach:.0%}, thickness {detach - attach:.0%})")

Capital structure (fractions of pool notional):
  equity         attach=0.0000 detach=0.0300  (0%-3%, thickness 3%)
  mezz           attach=0.0300 detach=0.0700  (3%-7%, thickness 4%)
  senior         attach=0.0700 detach=0.1500  (7%-15%, thickness 8%)
  super_senior   attach=0.1500 detach=1.0000  (15%-100%, thickness 85%)


## Examples

Homogeneous pool: **200** names, PD **2.5%**, asset correlation **15%**, LGD from **40%** recovery, and **10,000** deterministic paths, against the equity/mezz/senior/super-senior stack defined above.

Run the two-factor Gaussian finite-pool simulation, report pool EL/VaR/ES, then ask the result for per-tranche statistics. `tranche_loss_statistics` returns expected loss (fraction and amount), VaR and ES at the simulation's own confidence, the breach probability, and the probability of a full write-down — quantities that are awkward to reconstruct from an expected-loss average alone.

In [3]:
n_pool = 200
pd_pool = 0.025
rho_asset = 0.15
lgd_pool = rec.lgd
name_notional = 1_000_000.0
total_notional = n_pool * name_notional
beta = rho_asset**0.5
credit_loadings = [beta * tf.cholesky_l10, beta * tf.cholesky_l11]

exposures = [
    CreditExposure(
        id=f"loan_{index + 1}",
        notional=name_notional,
        default_probability=pd_pool,
        lgd=lgd_pool,
        factor_loadings=credit_loadings,
    )
    for index in range(n_pool)
]
result = simulate_portfolio_loss(
    exposures,
    PortfolioLossConfig(10_000, 42, 0.99, CopulaSpec.gaussian()),
)

print(f"Pool: n={n_pool}, PD={pd_pool}, rho={rho_asset}, LGD={lgd_pool:.2f}, loadings={credit_loadings}")
print(
    f"Pool EL={result.expected_loss / total_notional:.4%}, "
    f"VaR99={result.var / total_notional:.4%}, "
    f"ES99={result.expected_shortfall / total_notional:.4%}"
)

# Rust owns the attach/detach allocation; attachment and detachment are fractions in [0, 1].
tranche_stats = {
    label: result.tranche_loss_statistics(attach, detach, total_notional) for label, attach, detach in tranches
}

header = f"{'tranche':<14}{'range':>12}{'notional':>16}{'EL':>10}{'EL amt':>14}{'VaR99':>10}{'ES99':>10}{'P(breach)':>11}{'P(wipeout)':>12}"
print("\n" + header)
print("-" * len(header))
for label, stats in tranche_stats.items():
    band = f"{stats.attachment:.0%}-{stats.detachment:.0%}"
    print(
        f"{label:<14}{band:>12}{stats.tranche_notional:>16,.0f}"
        f"{stats.expected_loss_fraction:>10.4%}{stats.expected_loss_amount:>14,.0f}"
        f"{stats.var_fraction:>10.2%}{stats.expected_shortfall_fraction:>10.2%}"
        f"{stats.prob_attachment_breached:>11.2%}{stats.prob_full_writedown:>12.2%}"
    )

Pool: n=200, PD=0.025, rho=0.15, LGD=0.60, loadings=[-0.07745966692414835, 0.3794733192202055]
Pool EL=1.4655%, VaR99=7.8000%, ES99=9.8079%

tranche              range        notional        EL        EL amt     VaR99      ES99  P(breach)  P(wipeout)
-------------------------------------------------------------------------------------------------------------
equity               0%-3%       6,000,000  41.3300%     2,479,800   100.00%   100.00%     85.40%      14.43%
mezz                 3%-7%       8,000,000   4.8900%       391,200   100.00%   100.00%     11.93%       1.38%
senior              7%-15%      16,000,000   0.3611%        57,780    10.00%    33.76%      1.38%       0.05%
super_senior      15%-100%     170,000,000   0.0013%         2,160     0.00%     0.13%      0.05%       0.00%


## Takeaways

- `LatentTwoFactor.clo_standard()` maps directly into two systematic exposure loadings $\sqrt{\rho}[L_{10},L_{11}]$.
- The canonical Rust engine owns finite-pool default simulation, deterministic RNG, loss aggregation, pool VaR/ES, **and** the attach/detach tranche allocation via `PortfolioLossResult.tranche_loss_statistics`.
- Attachment and detachment are fractions in $[0,1]$; a 0–3% tranche is `(0.0, 0.03)`.
- Subordination clips the same pool loss sequentially: equity is first-loss and super-senior is last-loss. `prob_attachment_breached` for one tranche therefore tracks `prob_full_writedown` of the tranche below it — exactly at the 7% and 15% boundaries above.
- The equity/mezz boundary is the exception (14.43% vs 11.93%): with 200 equal names at 60% LGD each default costs 0.3% of the pool, so pool loss lands on a discrete grid and the paths sitting *exactly* at 3% wipe out equity ($\ell \ge A$) without breaching mezz ($\ell > A$). Finite pools are lumpy — a large-pool approximation would hide this.
- Base correlation is a per-tranche implied parameter; one homogeneous asset correlation remains a pedagogical simplification.
- Pair this with `recovery_modeling.ipynb` for factor-dependent LGD.